#### **Requests**
The Requests libary is used to make HTTP requests in python. Though its being a third party tool (no in the python STL) we can use requests to perform HTTP actions like GET, POST, and more.

### **GET**
GET is a common HTTP method which retrieves data from a specified resource. to make a get request using `requests`, we can invoke `requests.get()`.

In [1]:
import requests
requests.get("https://api.github.com")

<Response [200]>

We got a response. A **response** is the object that contains the results of our requests. it ypically has a **status code** which tells us the status of our request. 

In [2]:
response = requests.get("https://api.github.com")
response.status_code

200

200 means the request was sucessfull and the server responded with the request we asked for.

you can use Request’s built-in capacities to raise an exception if the request was unsuccessful. You can do this using `.raise_for_status()` instead of a if-else statement:

In [4]:
from requests.exceptions import HTTPError

URLS = ["https://api.github.com", "https://api.github.com/invalid"]

for url in URLS:
    try:
        response = requests.get(url)
        response.raise_for_status()
    except HTTPError as http_err:
        print(f"HTTP error ocurred: {http_err}")
    except Exception as err:
        print(f"other error occured {err}")
    else:
        print('success')

success
HTTP error ocurred: 404 Client Error: Not Found for url: https://api.github.com/invalid


#### **Access the Response Content**

reponses of get request typically come with some `payload`, which is valuable information in the message body. we can view the payload in different formats

In [20]:
reponse = requests.get("https://api.github.com")
print(reponse.content)
print(type(response.content))

b'{\n  "current_user_url": "https://api.github.com/user",\n  "current_user_authorizations_html_url": "https://github.com/settings/connections/applications{/client_id}",\n  "authorizations_url": "https://api.github.com/authorizations",\n  "code_search_url": "https://api.github.com/search/code?q={query}{&page,per_page,sort,order}",\n  "commit_search_url": "https://api.github.com/search/commits?q={query}{&page,per_page,sort,order}",\n  "emails_url": "https://api.github.com/user/emails",\n  "emojis_url": "https://api.github.com/emojis",\n  "events_url": "https://api.github.com/events",\n  "feeds_url": "https://api.github.com/feeds",\n  "followers_url": "https://api.github.com/user/followers",\n  "following_url": "https://api.github.com/user/following{/target}",\n  "gists_url": "https://api.github.com/gists{/gist_id}",\n  "hub_url": "https://api.github.com/hub",\n  "issue_search_url": "https://api.github.com/search/issues?q={query}{&page,per_page,sort,order}",\n  "issues_url": "https://api.

In [21]:
# as text
response.text

'{"message":"Not Found","documentation_url":"https://docs.github.com/rest","status":"404"}'

In [12]:
type(response.text)

str

Because the decoding of bytes to a str requires an encoding scheme, Requests will try to guess the encoding based on the response’s headers if you don’t specify one. You can provide an explicit encoding by setting `.encoding` before accessing `.text`:

In [13]:
response.encoding = "utf-8"  # Optional: Requests infers this.
response.text

'{"message":"Not Found","documentation_url":"https://docs.github.com/rest","status":"404"}'

we can notice that the response we see is actually a serialized `JSON` content. To get a dictionary format from this we can alsways retrieve the `str` and deserialize this using `json.load()` but a quicker way to do this is to use `.json()`

In [14]:
response.json()

{'message': 'Not Found',
 'documentation_url': 'https://docs.github.com/rest',
 'status': '404'}

In [15]:
type(response.json())

dict

Now we have it in a dictionary format and we can easily manipulate or extract relevant info form this payload.

In [22]:
response_dict = response.json()
print(response_dict['documentation_url'])

https://docs.github.com/rest


#### **Other HTTP Methods**

other popular HTTP methods include POST, PUT, DELETE, HEAD, PATCH, and OPTIONS.

> To try out these HTTP methods, you’ll make requests to httpbin.org. The httpbin service is a great resource created by the original author of Requests, Kenneth Reitz. The service accepts test requests and responds with data about the requests.



In [23]:
import requests 

requests.get("https://httpbin.org/get")

<Response [200]>

In [26]:
requests.post("https://httpbin.org/post", data= {"key":"value"})

<Response [200]>

In [27]:
requests.put("https://httpbin.org/put", data= {"key": "value"})

<Response [200]>

In [28]:
requests.delete("https://httpbin.org/delete")


<Response [200]>

In [29]:
requests.head("https://httpbin.org/get")


<Response [200]>

All these methods are high-level shortcuts to `request.request()`

In [30]:
requests.request("GET", "https://httpbin.org/get")

<Response [200]>

In [31]:
response = requests.head("https://httpbin.org/get")

In [32]:
response.headers

{'Date': 'Thu, 24 Jul 2025 09:06:05 GMT', 'Content-Type': 'application/json', 'Content-Length': '309', 'Connection': 'keep-alive', 'Server': 'gunicorn/19.9.0', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Credentials': 'true'}

In [33]:
response.headers['content-type']

'application/json'

When we make a request, the Requests libary prepares the request before actually sending it to the destination server. By preparing the request this means tasks like validating headers and serializing json.

#### **Inspecting the prepared requests**

you can view PreparedRequest object by accessing `.request`

In [34]:
import requests

response = requests.post("https://httpbin.org/post", json={"key":"value"})

In [35]:
response.request

<PreparedRequest [POST]>

In [36]:
response.request.headers['Content-type']

'application/json'

In [37]:
response.request.url

'https://httpbin.org/post'

In [38]:
response.request.body

b'{"key": "value"}'

#### **Use Authentication**

**Why is HTTP not secure?**

when you send a request containing your credentials to the server (username and password). your request sets a string prefixed with `Basic`  as your Authorization header. The HTTP request combines your username and password, inserting a column between them i.e `user:passwd` and encodes this string in Base64 using `base64.b64encode()`. The encoding converts the string to another string (i.e `dXNlcjpwYXNzd2Q=`) and places `Basic` as a prefix to the encoded string. This is very easy easy to decode, hence revielimngm your credential this is why HTTP isn't secire adn request should be sent over HTTPS, which encrypts the entire request and provides an additional layer of protection.

To specify our server credentials while making a request we can use the `auth` key word:

In [39]:
response = requests.get("https://httpbin.org/basic-auth/user/passwd",
                        auth= ("user", "passwd"))
# my credentials here are username: "user", password: "passwd"

In [40]:
response.status_code

200

In [41]:
response.request.headers["Authorization"]

'Basic dXNlcjpwYXNzd2Q='

Note: the recquest only succeeds if the credentials you pass as a tuple into the auth parameter are correct. **Request applies the HTTP basic access authentification scheme under the hood which is not secured**

explicitly:

In [44]:
# just for demonstration we don't have to do this
from requests.auth import HTTPBasicAuth
requests.get("https://httpbin.org/basic-auth/user/passwd",
        auth=HTTPBasicAuth("user", "passwd"))


<Response [200]>

In real life some APIs require Authentification and we need to generate a personal access token which can be use inplace of a username and password (if you encouneter such case make reference to the API documentation to know how to generate token) but code typically looks like this

```python
import requests
token = "blablabla"
url = "https://......"
response = requests.get(url, auth = ("", token))

# do anythng you want
response.status_code
```

as we learnt before the above approach using HTTP Authentication which is less secure. this works bu this is not the right way to authenticate with a bearer token. We can supply our own authentication mechanism to fix this. say we want to use `Authbase` or `OAuth` we can write a custom authentication class to attach the token to the choosen Authentification schemes. and our authentication class would act as a middle man encapsulating our token so it isn't exposed unnecessarily to th client or server, making it less likely to be intercepted by attackers. Additionally to avoid vulnerability we can use HTTPS instead of HTTP to ensure the the token is only used in secure communication (ie over HTTPS) by setting up and SSL/TLS and simply replacing `http://` with `https://` in our request url.

In [2]:
from requests.auth import AuthBase

class TokenAuth(AuthBase):
    """Implement a token authentication scheme"""
    
    def __init__(self, token):
        self.token = token

    def __call__(self, request):
        """Attach an API token to Authorization header."""
        request.headers["Authorization"] = f"Bearer {self.token}"
        return request

In [ ]:
token = "blablabla"
url = "https://........"
response = requests.get(
    url,
    auth=TokenAuth(token)
)
print(response.status_code) #  should return status code
print(response.request.headers["Authorization"]) # should return Auth token

#### **Communicating securely with servers**
